# 01 — EDA: Kurs USD/IDR & Berita CNBC

Tujuan: memahami distribusi kurs, liputan berita, dan hubungan awal sentimen vs pergerakan kurs **sebelum** modeling.
Sumber: `data/dateadjusted/rate.csv` + `data/dateadjusted/cnbc.csv` + `data/splits/`.
Jalankan dari folder `notebook/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rate = pd.read_csv('../data/dateadjusted/rate.csv', parse_dates=['tanggal'])
news = pd.read_csv('../data/dateadjusted/cnbc.csv')
news['adjusted_date'] = pd.to_datetime(news['adjusted_date'])
print(rate.shape, news.shape)
print('kurs:', rate['tanggal'].min(), '->', rate['tanggal'].max())
print('berita:', news['adjusted_date'].min(), '->', news['adjusted_date'].max())

## Temuan (precomputed, 30 Sep 2026)

- **Kurs**: 1202 hari trading, 2021-09-01 → 2026-09-01, Rp14.284 → Rp17.727 (tren melemah; rata-rata train ≈ Rp15.279, test ≈ Rp17.310).
- **Berita**: 8679 artikel pada 1009 hari unik; rata-rata 8,6/hari, median 4, maks 166 (distribusi sangat skew — hari krisis membanjiri liputan).
- **Panjang teks**: median judul 79 karakter; median isi 564 kata (isi kaya tapi noisy).
- **Hari tanpa berita**: ±193 dari 1202 hari trading → pipeline mengisi fitur sentimen netral/nol (`news_count=0`).
- **Berita di luar rentang kurs** (adjusted_date > 2026-09-01, ±172 artikel) dibuang karena tidak punya target.

In [ ]:
# Tren kurs + return harian
rate = rate.sort_values('tanggal')
rate['ret'] = rate['kurs'].pct_change()
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(rate['tanggal'], rate['kurs'])
ax[0].set_title('USD/IDR JISDOR (2021-2026)')
ax[0].set_ylabel('Rp per USD')
ax[1].plot(rate['tanggal'], rate['ret'])
ax[1].set_title('Return harian')
ax[1].set_ylabel('return')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi jumlah berita per hari trading
cnt = news['adjusted_date'].value_counts()
print(cnt.describe().round(2).to_string())
plt.figure(figsize=(8, 4))
plt.hist(cnt.values, bins=40)
plt.title('Distribusi jumlah artikel per hari')
plt.xlabel('artikel/hari'); plt.ylabel('hari')
plt.show()

## Target klasifikasi

`direction = 1` (NAIK) bila kurs besok > kurs hari ini. Setelah warm-up fitur teknikal (MA-20), tersisa **1181 hari**: train 826 (2021-09-29→2025-02-25), val 177, test 178.
Proporsi NAIK: train **55,8%** / val **52,0%** / test **57,9%** — sedikit condong positif mengikuti tren depresiasi, sehingga *naive mayoritas* bukan baseline yang sepele (akurasi test 0,579 dengan F1-macro hanya 0,367).

In [ ]:
# Sentimen harian (judul, VADER compound mean) vs return besok
splits = {s: pd.read_csv(f'../data/splits/{s}.csv') for s in ('train', 'val', 'test')}
for s, df in splits.items():
    c = df['t_vader_compound_mean'].corr(df['ret_next'])
    print(f'{s}: corr(VADER-judul, return-besok) = {c:.4f} | news/hari = {df["t_news_count"].mean():.2f}')

plt.figure(figsize=(8, 4))
plt.scatter(splits['train']['t_vader_compound_mean'], splits['train']['ret_next'], s=5, alpha=0.4)
plt.title('Sentimen judul harian vs return besok (train)')
plt.xlabel('mean VADER compound judul'); plt.ylabel('return besok')
plt.show()

## Implikasi untuk modeling

1. Korelasi linear sentimen→return ≈ **+0,04 (sangat lemah)** — model linear kurs-saja diprediksi ≈ acak; nilai tambah NLP harus diuji via ablation, bukan asumsi.
2. Liputan didominasi topik AS umum (bukan khusus IDR) — filtering topik adalah next step yang paling menjanjikan.
3. Ketidakseimbangan liputan (1 vs 166 artikel/hari) membenarkan agregasi `mean/std/min/max/count` alih-alih satu rata-rata.